# Copy validation images
Copies all images with `is_train == 0` from the metadata CSV into a destination folder.

Update the paths in the next cell before running.

In [2]:
from pathlib import Path

# Path to the CSV with the is_train column
csv_path = Path(r"f:\InfTech\Prodotti\Python\GeoLocGit\GeoLoc-CVCS\data\metadata\s2-geo-cells\train_val_split_geocells_total_expanded.csv")

# Root folder where images live
images_root = Path(r"f:\InfTech\Prodotti\Python\GeoLocGit\GeoLoc-CVCS\data\raw")

# Destination folder for validation images
dest_root = Path(r"f:\InfTech\Prodotti\Python\GeoLocGit\GeoLoc-CVCS\data\splits\validation_images")

# Common image extensions to try when the CSV id has no extension
extensions = [".jpg", ".jpeg", ".png", ".webp"]

# If True, only reports counts without copying
dry_run = False

In [ ]:
from typing import Optional
import pandas as pd
import shutil

def resolve_image_path(rel_id: str) -> Optional[Path]:
    rel_path = Path(rel_id)
    # If rel_id already contains an extension, check directly
    if rel_path.suffix:
        candidate = images_root / rel_path
        return candidate if candidate.exists() else None

    # Otherwise, try common extensions
    for ext in extensions:
        candidate = images_root / rel_path.with_suffix(ext)
        if candidate.exists():
            return candidate
    return None

df = pd.read_csv(csv_path)
val_df = df[df["is_train"] == 0].copy()

# Clean image names to only keep the image ID (last part of path)
val_df["image_id"] = val_df["id"].apply(lambda x: Path(x).stem)

missing = []
copied = 0

for rel_id in val_df["id"].astype(str):
    src = resolve_image_path(rel_id)
    if src is None:
        missing.append(rel_id)
        continue

    dest = dest_root / src.relative_to(images_root)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dry_run:
        shutil.copy2(src, dest)
    copied += 1

print(f"Validation rows: {len(val_df)}")
print(f"Copied: {copied}")
print(f"Missing: {len(missing)}")

if missing:
    print("Sample missing IDs (first 10):")
    print(missing[:10])

Validation rows: 26584
Copied: 26584
Missing: 0


In [3]:
# create a csv with only the validation entries

val_csv_path = dest_root.parent / "validation_split.csv"
# Reorder columns to have image_id first after id
cols = ["id", "image_id"] + [c for c in val_df.columns if c not in ["id", "image_id"]]
val_df[cols].to_csv(val_csv_path, index=False)
print(f"Saved validation CSV to: {val_csv_path}")

Saved validation CSV to: f:\InfTech\Prodotti\Python\GeoLocGit\GeoLoc-CVCS\data\splits\validation_split.csv
